In [ ]:
!pip install transformers
!pip install torch
!pip install scikit-learn
!pip install matplotlib
!pip install numpy
!pip install seaborn
!pip install krippendorff

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from sklearn.model_selection import train_test_split
import torch.optim as optim
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, accuracy_score
)
import pandas as pd
import os
import io
import csv
import numpy as np
import re
import sys
import seaborn as sns
from tqdm import tqdm, trange
import matplotlib.pyplot as plt
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

In [ ]:
def clean_paragraph_text(text):
    """Decode HTML entities and remove paragraph tags from a text string.
    
    Returns the original value unchanged if it is NaN (missing).
    """
    if pd.isna(text):
        return text  # Preserve NaNs; they will be caught downstream if needed

    replacements = {
        "&eacute;": "é",
        "&agrave;": "à",
        "&egrave;": "è",
        "&Icirc;": "Î",
        "&ccedil;": "ç",
        "&acirc;": "â",
        "</P> <P>": " ",   # Merge adjacent paragraph blocks into a single string
    }

    for key, val in replacements.items():
        text = text.replace(key, val)

    return text

In [ ]:
MODEL_SAVE_PATH = "model_eurobert_political_1"

In [ ]:
# Load the full unlabelled corpus
all_interactions = pd.read_csv("flat_all_interactions_revert.csv", delimiter=',')
print(f"Total texts to classify: {len(all_interactions):,}")

In [ ]:
# Apply the same text cleaning used during training
all_interactions['texte'] = all_interactions['texte'].apply(clean_paragraph_text)

In [ ]:
# Split the full dataset into 20 chunks of roughly equal size
# This limits memory usage per iteration and allows intermediate saves
N_CHUNKS = 20
parts = np.array_split(all_interactions, N_CHUNKS)
print(f"Chunk sizes: {[len(p) for p in parts]}")

In [ ]:
# Load the saved model and tokenizer for inference
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_SAVE_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_SAVE_PATH)

In [ ]:
# Re-detect device and move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    print(f"Using GPU for inference: {torch.cuda.get_device_name(0)}")
else:
    print("GPU not available. Using CPU — inference will be slower.")

model.to(device)

In [ ]:
INFERENCE_BATCH_SIZE = 256 #depend on the GPU VRAM

for i in range(2, N_CHUNKS + 1):
    print(f"\n── Chunk {i}/{N_CHUNKS} ──")
    chunk_text = parts[i - 1].copy()

    # EuroBERT supports up to 8192 tokens — raise max_length if paragraphs are long
    encodings = tokenizer(
        chunk_text.ravel().tolist(),
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors='pt'
    )
    dataset    = TensorDataset(encodings['input_ids'], encodings['attention_mask'])
    dataloader = DataLoader(dataset, batch_size=INFERENCE_BATCH_SIZE)

    print('Data loaded')
    model.eval()
    all_preds = []

    j = 0
    for batch in dataloader:
        print(j)
        input_ids_b, attention_mask_b = tuple(t.to(device) for t in batch)
        with torch.no_grad():
            outputs = model(input_ids_b, attention_mask=attention_mask_b)
            logits  = outputs.logits               # shape: (batch_size, 2)
            preds   = torch.argmax(logits, dim=1)  # shape: (batch_size,)
            all_preds.extend(preds.cpu().numpy())

        j+=1

    df = pd.DataFrame(chunk_text, columns=['texte'])
    df['political'] = all_preds
    df_positive = df[df['political'] == 1].copy()

    # Bug fix: typo 'apply mode' → 'apply model'
    out_path = f'output/data_political_{i}.csv'
    df_positive.to_csv(out_path, index=False)
    print(f"  Classified: {len(df)} rows ; {len(df_positive)} positive ; saved to {out_path}")

In [ ]:
# Merge in final file
list_outputs = os.listdir('output')
first_file = list_outputs.pop(0)

df = pd.read_csv(first_file)
for file in list_outputs:
    temp = pd.read_csv(file)
    df = pd.concat([df, temp])
df = df.drop_duplicates()
df = df.rename(columns={'texte':'text'})
df = df.drop(columns=['political'])
df.to_csv('flat_political_interactions.csv', index=False)
print(len(df))